# Phase 5 — Smoke Tests : Spatial Residual Encoder (SRE)

## Architecture dual-branch Stage 1
```
LR field [B,C,H_lr,W_lr]          DAG nodes (pooled)
        │                                │
  [SRE — CNN + AdaLN]          [GNN + RCN + head]
  conditionné par H_T →               │
        │                          μ_HR_causal
        │ δ_spatial                    │
        └──────────── + ───────────────┘
                       │
                  μ_HR_total  →  [Stage 2 diffusion]
```

## Consensus expert (Phase 5, 2026-06-19)
| Décision | Justification |
|---|---|
| Zero-init out_head + out_gain=0.1 | δ≈0 à t=0, μ_causal domine au démarrage |
| AdaLN injecté à chaque bloc | DAG contrôle quels patterns spatiaux le SRE amplifie |
| λ_causal·MSE(μ_causal, HR) en loss auxiliaire | Identifiabilité DAG (CaPaint 2409.19608) |
| Freeze DAG+GNN+RCN+head d'abord | Q_phys=0.9998 ne doit pas bouger |
| ≤200k params SRE | Évite que SRE devienne un downscaler standalone |
| Q_phys évalué sur μ_causal seul | Préserve la validité de la probe Phase A1 |

## 8 smoke tests
| # | Test | Invariant |
|---|---|---|
| T1 | Shapes | μ_total = μ_causal + δ ∈ [B,1,172,179] |
| T2 | Zero-init | \|δ\|_max < 1e-4 à l'init |
| T3 | Gradient flow | Gradients non-nuls dans SRE + encoder + out_gain |
| T4 | DAG freeze | A_dag.grad = None quand requires_grad=False |
| T5 | AdaLN effectif | Même LR, H_T différent → δ différent |
| T6 | NaN océan | lr_field NaN → δ NaN-free après nan_to_num |
| T7 | stage1_compute_loss | Loss finie + grad reach out_gain |
| T8 | μ_causal_frac ≈ 1.0 | SRE silencieux à l'init |


In [ ]:
# === Cell 1 : Bootstrap ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; from omegaconf import OmegaConf; import diffusers
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')

In [ ]:
# === Cell 2 : Imports + constantes ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from st_cdgm.models.spatial_residual_encoder import SpatialResidualEncoder, AdaLNConvBlock
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.training.two_stage import stage1_compute_loss

# --- Shapes (synthétiques, rapides sur CPU) ---
B       = 2       # batch
C_LR    = 10      # canaux LR (q_500, q_850, u_500, u_850, v_500, v_850, t_500, t_850 + 2)
H_LR    = 22      # grille LR hauteur
W_LR    = 23      # grille LR largeur
H_HR    = 172     # grille HR hauteur
W_HR    = 179     # grille HR largeur
Q       = 9       # nœuds DAG (9-node)
N_NODES = 50      # nœuds GNN par type (synthétique; réel ≈ 333-506)
D       = 128     # hidden dim RCN / conditioning dim
SEED    = 42
DEV     = torch.device('cpu')

# --- Seuils smoke tests ---
ZERO_INIT_THR   = 1e-4    # T2 : |δ|_max à l'init
ADALN_EPS       = 1e-3    # T5 : diff minimale quand H_T change
CAUSAL_FRAC_TOL = 1e-3    # T8 : |frac - 1.0| tolérance

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'[Cell 2] Shapes : LR={B}×{C_LR}×{H_LR}×{W_LR}  HR=172×179  Q={Q} N={N_NODES} D={D}')
print(f'[Cell 2] Device : {DEV}')

In [ ]:
# === Cell 3 : Helpers — données synthétiques + stack ===

def _seed(s=SEED):
    torch.manual_seed(s); np.random.seed(s)

def _fake_HT(requires_grad=False, q=Q, n=N_NODES):
    """H_T shape [B, q, N, D] — sortie typique de RCNSequenceRunner."""
    t = torch.randn(B, q, n, D, device=DEV) * 0.1
    return t.requires_grad_(requires_grad)

def _fake_lr(nan_ocean=False):
    x = torch.randn(B, C_LR, H_LR, W_LR, device=DEV)
    if nan_ocean:
        x[:, :, :3, :3] = float('nan')    # patch océan simulé
    return x

def _fake_mask():
    m = torch.ones(B, 1, H_HR, W_HR, device=DEV)
    m[:, :, :20, :20] = 0.0   # zone océan
    return m

def _build_sre():
    _seed()
    return SpatialResidualEncoder(
        in_channels=C_LR, d_cond=D, base_ch=32,
        hr_h=H_HR, hr_w=W_HR, out_gain=0.1,
    ).to(DEV)

def _build_full_stack():
    """Build SRE + GraphToGridDecoder + RCNCell (no GNN encoder — uses fake H_T)."""
    _seed()
    sre  = _build_sre()
    head = GraphToGridDecoder(
        d_model=D, hr_h=H_HR, hr_w=W_HR,
        intermediate_h=43, intermediate_w=45,
        n_heads=4, refine_channels=64, output_channels=1,
    ).to(DEV)
    rcn  = RCNCell(
        num_vars=Q, hidden_dim=D,
        driver_dim=C_LR, reconstruction_dim=C_LR,
    ).to(DEV)
    return sre, head, rcn

# Vérification que le module est bien importé
sre_test = _build_sre()
print(f'[Cell 3] SRE instancié : {sre_test.num_params():,} params')
assert sre_test.num_params() < 500_000, 'SRE > 500k params — risque de standalone downscaler'
print('[Cell 3] Contrainte capacité OK (<500k params)')

In [ ]:
# === T1 : Shapes ===
def test_t1_shapes():
    sre, head, _ = _build_full_stack()
    H_T    = _fake_HT()
    lr     = _fake_lr()

    mu_c   = head(H_T)                          # branche causale
    delta  = sre(torch.nan_to_num(lr), H_T)     # branche spatiale
    mu_tot = mu_c + delta                        # fusion additive

    exp = (B, 1, H_HR, W_HR)
    assert tuple(mu_c.shape)   == exp, f'mu_causal {mu_c.shape} ≠ {exp}'
    assert tuple(delta.shape)  == exp, f'delta {delta.shape} ≠ {exp}'
    assert tuple(mu_tot.shape) == exp, f'mu_total {mu_tot.shape} ≠ {exp}'
    print(f'[T1] ✓ SHAPES OK  mu_total={tuple(mu_tot.shape)}')

test_t1_shapes()

In [ ]:
# === T2 : Zero-init — δ ≈ 0 à l'initialisation ===
def test_t2_zero_init():
    sre = _build_sre()
    H_T = _fake_HT()
    lr  = torch.nan_to_num(_fake_lr())

    with torch.no_grad():
        delta = sre(lr, H_T)

    amax = delta.abs().max().item()
    assert amax < ZERO_INIT_THR, (
        f'|δ|_max={amax:.3e} ≥ {ZERO_INIT_THR} — '
        f'out_head non zero-init ou out_gain trop grand'
    )
    print(f'[T2] ✓ ZERO-INIT OK  |δ|_max={amax:.2e}  (seuil={ZERO_INIT_THR})')

test_t2_zero_init()

In [ ]:
# === T3 : Gradient flow — tous les sous-modules reçoivent un gradient ===
def test_t3_gradient_flow():
    sre, head, _ = _build_full_stack()

    # Casser le zero-init pour que out_gain.grad ne soit pas 0
    with torch.no_grad():
        sre.out_head[0].weight.normal_(0, 0.01)

    H_T = _fake_HT(requires_grad=True)
    lr  = torch.nan_to_num(_fake_lr()).requires_grad_(True)
    tgt = torch.randn(B, 1, H_HR, W_HR, device=DEV)

    mu_tot = head(H_T) + sre(lr, H_T)
    F.mse_loss(mu_tot, tgt).backward()

    # Vérifier gradients dans chaque module
    checks = {
        'head': head,
        'sre':  sre,
    }
    for name, mod in checks.items():
        grads = [p.grad for p in mod.parameters() if p.grad is not None]
        assert grads,                                   f'{name}: aucun gradient'
        assert not any(g.isnan().any() for g in grads), f'{name}: gradient NaN'
        assert any(g.abs().sum() > 0  for g in grads), f'{name}: gradient nul partout'

    # out_gain doit avoir un gradient
    assert sre.out_gain.grad is not None, 'out_gain: pas de gradient'
    assert not sre.out_gain.grad.isnan(),  'out_gain: gradient NaN'

    # H_T doit recevoir un gradient (le chemin RCN→head ET RCN→SRE doivent être actifs)
    assert H_T.grad is not None,            'H_T: pas de gradient (chemin RCN→SRE cassé?)'
    assert not H_T.grad.isnan().any(),      'H_T: gradient NaN'

    print(f'[T3] ✓ GRADIENT FLOW OK  out_gain.grad={sre.out_gain.grad.item():.3e}')

test_t3_gradient_flow()

In [ ]:
# === T4 : DAG freeze — A_dag ne reçoit pas de gradient ===
def test_t4_dag_freeze():
    sre, head, rcn = _build_full_stack()

    # Geler A_dag comme dans le fine-tuning Phase 4/5
    rcn.A_dag.requires_grad_(False)
    A_dag_before = rcn.A_dag.detach().clone()

    H_T  = _fake_HT(requires_grad=True)
    lr   = torch.nan_to_num(_fake_lr())
    mu_tot = head(H_T) + sre(lr, H_T)
    mu_tot.pow(2).mean().backward()

    # A_dag.grad doit être None ou 0
    grad_sum = 0.0 if rcn.A_dag.grad is None else rcn.A_dag.grad.abs().sum().item()
    assert grad_sum == 0.0, f'A_dag.grad sum={grad_sum:.3e} ≠ 0 (freeze cassé!)'

    # A_dag ne doit pas avoir bougé
    drift = (rcn.A_dag - A_dag_before).abs().max().item()
    assert drift == 0.0, f'A_dag a dérivé de {drift:.3e} malgré le freeze'

    print(f'[T4] ✓ DAG FREEZE OK  A_dag.grad_sum={grad_sum:.1e}  drift={drift:.1e}')

test_t4_dag_freeze()

In [ ]:
# === T5 : AdaLN conditioning effectif — H_T différent → δ différent ===
def test_t5_adaln_conditioning():
    sre = _build_sre()

    # Casser le zero-init pour que le conditioning soit visible
    with torch.no_grad():
        sre.out_head[0].weight.normal_(0, 0.02)

    lr = torch.nan_to_num(_fake_lr())

    torch.manual_seed(1); H_T1 = _fake_HT()
    torch.manual_seed(2); H_T2 = _fake_HT()

    with torch.no_grad():
        d1 = sre(lr, H_T1)
        d2 = sre(lr, H_T2)

    max_diff = (d1 - d2).abs().max().item()
    assert max_diff > ADALN_EPS, (
        f'AdaLN inerte : max_diff={max_diff:.2e} ≤ {ADALN_EPS}. '
        f'Le SRE ignore le conditioning DAG.'
    )
    print(f'[T5] ✓ ADALN OK  max_diff={max_diff:.3e}  (seuil>{ADALN_EPS})')

test_t5_adaln_conditioning()

In [ ]:
# === T6 : NaN océan — nan_to_num avant SRE → δ NaN-free ===
def test_t6_nan_handling():
    sre = _build_sre()
    H_T = _fake_HT()

    lr_nan  = _fake_lr(nan_ocean=True)
    n_nan   = lr_nan.isnan().sum().item()
    lr_safe = torch.nan_to_num(lr_nan, nan=0.0)   # convention standard

    with torch.no_grad():
        delta = sre(lr_safe, H_T)

    assert not delta.isnan().any(), f'NaN dans δ malgré nan_to_num (pixels océan : {n_nan})'
    assert not delta.isinf().any(), f'Inf dans δ'
    print(f'[T6] ✓ NaN HANDLING OK  (lr avait {n_nan} NaN, δ propre)')

test_t6_nan_handling()

In [ ]:
# === T7 : stage1_compute_loss intégration ===
# Vérifie que la loss est finie et que le gradient atteint SRE.out_gain
def test_t7_stage1_loss():
    sre, head, _ = _build_full_stack()

    # Casser zero-init pour avoir un gradient non-nul sur out_gain
    with torch.no_grad():
        sre.out_head[0].weight.normal_(0, 0.01)

    H_T  = _fake_HT(requires_grad=True)
    lr   = torch.nan_to_num(_fake_lr())
    mask = _fake_mask()
    tgt  = torch.randn(B, 1, H_HR, W_HR, device=DEV)

    mu_c   = head(H_T)
    delta  = sre(lr, H_T)
    mu_tot = mu_c + delta

    # Loss principale sur μ_total
    loss_total, logs = stage1_compute_loss(
        mu_HR=mu_tot,
        target_residual=tgt,
        valid_mask=mask,
        lambda_reg=1.0,
    )

    # Loss causale auxiliaire (Expert 2 : λ_causal · MSE(μ_causal, HR_true))
    LAMBDA_CAUSAL = 0.7   # valeur warm-up (schedule 1.0 → 0.3)
    loss_causal   = F.mse_loss(mu_c[mask.bool()], tgt[mask.bool()])
    loss          = loss_total + LAMBDA_CAUSAL * loss_causal

    # Logging diagnostique
    logs['delta_std']      = delta.detach().std().item()
    logs['mu_causal_std']  = mu_c.detach().std().item()
    logs['loss_causal']    = loss_causal.item()
    logs['lambda_causal']  = LAMBDA_CAUSAL

    assert torch.isfinite(loss), f'Loss non finie : {loss.item()}'
    loss.backward()

    assert sre.out_gain.grad is not None, 'out_gain : pas de gradient via stage1_loss'
    assert not sre.out_gain.grad.isnan(), 'out_gain.grad : NaN'

    print(f'[T7] ✓ STAGE1-LOSS OK')
    print(f'       loss_total={loss_total.item():.4f}  loss_causal={loss_causal.item():.4f}')
    print(f'       delta_std={logs["delta_std"]:.4f}  mu_causal_std={logs["mu_causal_std"]:.4f}')

test_t7_stage1_loss()

In [ ]:
# === T8 : μ_causal_frac ≈ 1.0 à l'init (SRE silencieux) ===
def test_t8_mu_causal_frac():
    sre, head, _ = _build_full_stack()   # SRE toujours zero-init ici
    H_T = _fake_HT()
    lr  = torch.nan_to_num(_fake_lr())

    with torch.no_grad():
        mu_c  = head(H_T)
        delta = sre(lr, H_T)

    num  = mu_c.abs().mean()
    den  = (mu_c.abs() + delta.abs()).mean().clamp_min(1e-12)
    frac = (num / den).item()

    assert abs(frac - 1.0) < CAUSAL_FRAC_TOL, (
        f'μ_causal_frac={frac:.6f} ≠ 1.0 (tolérance={CAUSAL_FRAC_TOL}). '
        f'SRE contribue déjà — zero-init cassé ?'
    )
    print(f'[T8] ✓ μ_CAUSAL_FRAC OK  frac={frac:.6f}  (seuil |1-frac| < {CAUSAL_FRAC_TOL})')

test_t8_mu_causal_frac()

In [ ]:
# === Cell finale : run_all + verdict ===
import traceback

TESTS = [
    ('T1 Shapes',            test_t1_shapes),
    ('T2 Zero-init',         test_t2_zero_init),
    ('T3 Gradient flow',     test_t3_gradient_flow),
    ('T4 DAG freeze',        test_t4_dag_freeze),
    ('T5 AdaLN conditioning',test_t5_adaln_conditioning),
    ('T6 NaN handling',      test_t6_nan_handling),
    ('T7 stage1_loss',       test_t7_stage1_loss),
    ('T8 μ_causal_frac',     test_t8_mu_causal_frac),
]

results = {}
print('=' * 60)
print('PHASE 5 — SRE SMOKE TESTS')
print('=' * 60)
for name, fn in TESTS:
    try:
        fn()
        results[name] = 'PASS'
    except Exception as e:
        results[name] = f'FAIL: {e}'
        traceback.print_exc()

print('\n' + '=' * 60)
passed = sum(1 for v in results.values() if v == 'PASS')
total  = len(results)
for name, status in results.items():
    icon = '✓' if status == 'PASS' else '✗'
    print(f'  {icon} {name:28s} {status}')
print('=' * 60)
if passed == total:
    print(f'\n✓ ALL {total}/{total} SMOKE TESTS PASSED')
    print('SRE prêt pour intégration dans train_epoch_stage1.')
    print('Prochaine étape : phase5_finetune_sre.ipynb')
    print('  — Freeze DAG+GNN+RCN+head')
    print('  — Train SRE sur résidu (HR_true - mu_causal.detach())')
    print('  — λ_causal schedule 1.0 → 0.3 sur 5 epochs')
else:
    print(f'\n✗ {total - passed}/{total} TESTS EN ÉCHEC — corriger avant intégration.')